# Localization Dataset-Adaptive-vs-Full Rebuttal Analysis

This notebook is for the small matched localization comparison:

- 10 selected examples per model x environment bundle
- dataset adaptive localization used as the reference traces
- the same subset rerun only with exhaustive `full` localization
- quantitative summaries of:
  - dataset-adaptive probe coverage
  - dataset-adaptive-vs-full peak and boundary agreement
  - prevalence of gradual and multi-peak exhaustive traces
  - case-study curves

If the analysis CSVs are missing or stale, run the refresh cell below.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd().resolve()
SEARCH_ROOTS = [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
REPO_ROOT = next(
    (
        root
        for root in SEARCH_ROOTS
        if (root / "rebuttal").exists() and (root / "src" / "sentence_localization_batch.py").exists()
    ),
    NOTEBOOK_CWD,
)
RESULTS_ROOT = Path('/playpen-ssd/smerrill/deception2/rebuttal/results')
if not RESULTS_ROOT.exists():
    RESULTS_ROOT = REPO_ROOT / "rebuttal" / "results"
RUN_NAME = 'localization_fulltrace_vs_adaptive_rebuttal_v1'
RUN_ROOT = RESULTS_ROOT / RUN_NAME
ANALYSIS_ROOT = RUN_ROOT / "analysis"
FIGURES_ROOT = ANALYSIS_ROOT / "figures"
ANALYSIS_SCRIPT = REPO_ROOT / "rebuttal" / "scripts" / "analyze_localization_fulltrace_rebuttal.py"

pd.options.display.max_columns = 200
pd.options.display.max_colwidth = 220
pd.options.display.width = 240

print("REPO_ROOT:", REPO_ROOT)
print("RUN_ROOT:", RUN_ROOT)
print("ANALYSIS_ROOT:", ANALYSIS_ROOT)


In [ ]:
def md(text: str) -> None:
    display(Markdown(text))


def read_json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def refresh_analysis() -> subprocess.CompletedProcess:
    cmd = [
        sys.executable,
        str(ANALYSIS_SCRIPT),
        "--run-name",
        RUN_NAME,
        "--results-root",
        str(RESULTS_ROOT),
    ]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
    return completed


completion_summary = read_json(ANALYSIS_ROOT / "completion_summary.json", default={}) or {}
selection_df = read_csv(RUN_ROOT / "selected_examples.csv")
bundle_selection_df = read_csv(RUN_ROOT / "bundle_summary.csv")
bundle_completion_df = read_csv(ANALYSIS_ROOT / "bundle_completion_summary.csv")
per_example_df = read_csv(ANALYSIS_ROOT / "per_example_metrics.csv")
curve_points_df = read_csv(ANALYSIS_ROOT / "curve_points.csv")
adaptive_bundle_df = read_csv(ANALYSIS_ROOT / "adaptive_vs_full_summary_by_bundle.csv")
adaptive_overall_df = read_csv(ANALYSIS_ROOT / "adaptive_vs_full_summary_overall.csv")
trace_shape_df = read_csv(ANALYSIS_ROOT / "trace_shape_prevalence_by_bundle.csv")
case_studies_df = read_csv(ANALYSIS_ROOT / "case_studies.csv")


## Refresh Analysis


In [ ]:
refresh = False
if refresh:
    refresh_analysis()
else:
    print("Set refresh = True and rerun this cell to recompute the analysis outputs.")


## Selection Inventory


In [ ]:
md(
    f"- Selected examples: **{len(selection_df)}**\n"
    f"- Bundles: **{bundle_selection_df.shape[0]}**"
)
bundle_selection_df


## Completion


In [ ]:
completion_summary


In [ ]:
bundle_completion_df


## Overall Adaptive-vs-Full Summary


In [ ]:
adaptive_overall_df


## Bundle Summary


In [ ]:
adaptive_bundle_df


In [ ]:
if not adaptive_bundle_df.empty:
    plot_df = adaptive_bundle_df.copy()
    plot_df["bundle_label"] = plot_df["env_display"].astype(str) + "\n" + plot_df["model_display"].astype(str)
    x = np.arange(len(plot_df))
    width = 0.32
    fig, ax = plt.subplots(figsize=(14.5, 5.4), constrained_layout=True)
    ax.bar(x - width / 2.0, plot_df["peak_within_one_rate"], width=width, label="Peak within one", color="#3D6E70")
    ax.bar(x + width / 2.0, plot_df["boundary_within_one_rate"], width=width, label="Boundary within one", color="#C9774D")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["bundle_label"], rotation=60)
    ax.set_ylim(0.0, 1.02)
    ax.set_ylabel("Agreement rate")
    ax.set_title("Dataset adaptive vs exhaustive agreement by bundle")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend()
    plt.show()
else:
    print("No paired adaptive/full results available yet.")


## Trace Shapes


In [ ]:
trace_shape_df


In [ ]:
if not trace_shape_df.empty:
    pivot = (
        trace_shape_df.pivot_table(
            index=["env_display", "model_display"],
            columns="trace_shape_label",
            values="fraction",
            fill_value=0.0,
        )
        .reset_index()
    )
    bundle_labels = pivot["env_display"].astype(str) + "\n" + pivot["model_display"].astype(str)
    categories = ["multi_peak", "gradual", "sharp_or_other"]
    colors = {
        "multi_peak": "#2B6CB0",
        "gradual": "#38A169",
        "sharp_or_other": "#D69E2E",
    }
    bottoms = np.zeros(len(pivot), dtype=float)
    fig, ax = plt.subplots(figsize=(14.5, 5.7), constrained_layout=True)
    for category in categories:
        values = pivot[category].to_numpy(dtype=float) if category in pivot.columns else np.zeros(len(pivot), dtype=float)
        ax.bar(bundle_labels, values, bottom=bottoms, color=colors[category], label=category.replace("_", " "))
        bottoms += values
    ax.set_ylabel("Fraction of exhaustive traces")
    ax.set_ylim(0.0, 1.02)
    ax.set_title("Gradual and multi-peak trace prevalence by bundle")
    ax.tick_params(axis="x", rotation=60)
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend()
    plt.show()
else:
    print("No exhaustive trace-shape outputs available yet.")


## Example-Level Metrics


In [ ]:
per_example_df.head(20)


## Case Studies


In [ ]:
case_studies_df


In [ ]:
def plot_example(bundle_key: str, example_id: str) -> None:
    subset = curve_points_df.loc[
        curve_points_df["bundle_key"].astype(str).eq(str(bundle_key))
        & curve_points_df["example_id"].astype(str).eq(str(example_id))
    ].copy()
    if subset.empty:
        print(f"No curve rows found for {bundle_key} / {example_id}")
        return
    full_df = subset.loc[subset["method"].astype(str).eq("full")].sort_values("sentence_idx")
    adaptive_df = subset.loc[subset["method"].astype(str).eq("adaptive")].sort_values("sentence_idx")
    metrics_row = per_example_df.loc[
        per_example_df["bundle_key"].astype(str).eq(str(bundle_key))
        & per_example_df["example_id"].astype(str).eq(str(example_id))
    ]
    if metrics_row.empty:
        print(f"No metric row found for {bundle_key} / {example_id}")
        return
    metric = metrics_row.iloc[0]
    fig, ax = plt.subplots(figsize=(10.5, 4.8), constrained_layout=True)
    ax.plot(full_df["sentence_number"], full_df["deception_rate"], marker="o", linewidth=2.4, color="black", label="Full")
    ax.scatter(adaptive_df["sentence_number"], adaptive_df["deception_rate"], s=70, color="#C05621", label="Dataset adaptive probes", zorder=4)

    peak_string = str(metric.get("full_prominent_peak_sentence_indices") or "")
    for peak_text in [value for value in peak_string.split(",") if value.strip()]:
        ax.axvline(int(peak_text), color="#3182CE", linewidth=1.1, alpha=0.35)

    if pd.notna(metric.get("adaptive_right_sentence_end_idx")):
        ax.axvline(int(metric["adaptive_right_sentence_end_idx"]), color="#DD6B20", linestyle="--", linewidth=1.4, alpha=0.75, label="Dataset adaptive right boundary")
    if pd.notna(metric.get("full_boundary_sentence_end_idx")):
        ax.axvline(int(metric["full_boundary_sentence_end_idx"]), color="#2F855A", linestyle=":", linewidth=1.6, alpha=0.85, label="Full boundary")

    ax.set_title(
        f"{metric['env_display']} / {metric['model_display']}\n"
        f"{metric['example_id']} | shape={metric['trace_shape_label']}"
    )
    ax.set_xlabel("Sentence index")
    ax.set_ylabel("Deception rate")
    ax.set_xticks(full_df["sentence_number"])
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
    plt.show()


if not case_studies_df.empty:
    first_case = case_studies_df.iloc[0]
    plot_example(first_case["bundle_key"], first_case["example_id"])
else:
    print("No case studies available yet.")


## Saved Figures


In [ ]:
if FIGURES_ROOT.exists():
    sorted(path.name for path in FIGURES_ROOT.glob("*.png"))
else:
    print("No figures directory found yet.")
